# BIOT 6900 · Module 8 Assignment — Single-Cell & Spatial Omics
## Starter notebook: one tumor, two views

This is your **starter**. The plumbing is done for you: the two datasets load in one line each, and the tricky steps (whole-donor sampling, scVI setup, Harmony, the spatial graph) are already written. Your job is to fill in the cells marked **`# TODO`** and to answer the **TODO** questions in the markdown.

- **Part 1 (single-cell):** which cell types express your target?
- **Part 2 (spatial):** where do those cells and that target sit in the tumor?
- **Part 3:** connect the two in your report.

How to use it: run top to bottom, completing each TODO as you reach it. The provided cells must run first for the TODO cells to work. When you are done, **Kernel → Restart & Run All** so the whole notebook executes cleanly, then commit it with your report.

> **Compute:** Part 1's scVI step wants a GPU. Run on Explorer via Open OnDemand (JupyterLab on `courses-gpu`, 1 GPU). Part 2 is CPU-only. See the lab guides for the steps.

### Requirements
Environment: the course `biot6900` / Explorer environment (`scanpy`, `cellxgene-census`, `scvi-tools`, `harmonypy`, `squidpy`). Needs **Python 3.13 or earlier**. If an import fails, run the install cell once and restart the kernel.

In [1]:
# Colab: run this cell first, then Runtime -> Restart session, then Run all (it will just say 'already satisfied').
!pip install "cellxgene-census" "scvi-tools" harmonypy squidpy scanpy leidenalg igraph


In [2]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.n_jobs = 1  # fix: Colab numba thread-count error in sc.pp.neighbors
sc.settings.set_figure_params(dpi=80, facecolor="white")
try:
    import torch
    GPU = torch.cuda.is_available()
    print("GPU available:", GPU)
except Exception:
    GPU = False
    print("running on CPU")
print("scanpy", sc.__version__, "| squidpy", sq.__version__)

ModuleNotFoundError: No module named 'scanpy'

---
# PART 1 · Single-cell: which cells express the target

## 1.1 · Pull the breast-cancer data  *(provided)*
This pulls breast-cancer blood/tumor cells from the CZ CELLxGENE Census and samples **10 whole donors**. You do not change anything here.

If the query returns zero cells on your Census version, replace the disease string with the exact one posted on Canvas (Census labels are version-specific).

In [3]:
import cellxgene_census

census = cellxgene_census.open_soma(census_version="stable")
obs = cellxgene_census.get_obs(
    census, "Homo sapiens",
    value_filter="disease == 'breast cancer' and is_primary_data == True",
    column_names=["soma_joinid", "donor_id", "assay", "cell_type"],
)
print(f"matching cells in the Census: {len(obs):,}")

# Sample whole donors (not scattered cells), so batch-aware steps have enough per donor.
rng = np.random.default_rng(0)
cells_per_donor = 2000
donor_sizes = obs["donor_id"].value_counts()
big_donors = donor_sizes[donor_sizes >= cells_per_donor].index.to_numpy()
donors = rng.choice(big_donors, size=min(10, len(big_donors)), replace=False)
sel = np.concatenate([
    rng.choice(obs.loc[obs["donor_id"] == d, "soma_joinid"].to_numpy(), size=cells_per_donor, replace=False)
    for d in donors
])
adata = cellxgene_census.get_anndata(
    census, "Homo sapiens", obs_coords=np.sort(sel),
    obs_column_names=["donor_id", "assay", "cell_type", "disease", "sex"],
    var_column_names=["feature_id", "feature_name"],
)
census.close()
adata.var_names = adata.var["feature_name"].astype(str)
adata.var_names_make_unique()

def broad_type(ct):
    ct = str(ct).lower()
    if "cd4" in ct or "cd8" in ct or "t cell" in ct or "thymocyte" in ct: return "T"
    if "plasma" in ct or "b cell" in ct: return "B"
    if "natural killer" in ct or "innate lymphoid" in ct: return "NK"
    if "monocyte" in ct: return "Monocyte"
    if "dendritic" in ct: return "DC"
    if "macrophage" in ct: return "Macrophage"
    if "epithel" in ct or "luminal" in ct or "basal" in ct or "carcinoma" in ct or "malignant" in ct or "tumor" in ct: return "Epithelial/Tumor"
    if "fibroblast" in ct or "stromal" in ct: return "Fibroblast"
    if "endothel" in ct: return "Endothelial"
    return "Other"
adata.obs["cell_type_broad"] = adata.obs["cell_type"].map(broad_type).astype("category")
print(f"\n{adata.n_obs:,} cells from {len(donors)} donors")
print(adata.obs["cell_type_broad"].value_counts())

ModuleNotFoundError: No module named 'cellxgene_census'

In [ ]:
# 1.1 Report cells, donors, and assays (required by the assignment)
print(f"Cells: {adata.n_obs:,} | Genes: {adata.n_vars:,} | Donors: {adata.obs['donor_id'].nunique()}")
print("\nAssays (chemistries):")
print(adata.obs['assay'].value_counts())
print("\nCells per donor x assay:")
print(pd.crosstab(adata.obs['donor_id'], adata.obs['assay']))
print("\nFine cell-type labels (top 15):")
print(adata.obs['cell_type'].value_counts().head(15))

## 1.2 · Standard pipeline  *(provided)*
QC, normalize, batch-aware highly variable genes, PCA. The raw counts are kept for scVI and for marker plots.

In [4]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)
adata.layers["counts"] = adata.X.copy()
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata = adata[(adata.obs.n_genes_by_counts < 6000) & (adata.obs.pct_counts_mt < 15)].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts", batch_key="donor_id")
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
print("cells after QC:", adata.n_obs)

NameError: name 'sc' is not defined

## 1.3 · The batch problem
The unintegrated embedding is computed for you. **TODO:** plot it and say what you see.

In [5]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca")
sc.tl.umap(adata)
adata.obsm["X_umap_unint"] = adata.obsm["X_umap"].copy()

NameError: name 'sc' is not defined

In [6]:
# 1.3 Unintegrated UMAP: colored by donor and by broad cell type
sc.pl.embedding(adata, basis="umap_unint", color=["donor_id", "cell_type_broad"], wspace=0.5)

NameError: name 'sc' is not defined

**TODO (interpretation):** In one or two sentences, what do these two plots show? Is the data organized by donor or by biology?

*Your answer here.*

## 1.4 · Integrate with scVI  *(provided, GPU step)*

In [7]:
import scvi
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="donor_id")
vae = scvi.model.SCVI(adata, n_latent=30)
vae.train(max_epochs=(200 if GPU else 60), accelerator=("gpu" if GPU else "cpu"))
adata.obsm["X_scVI"] = vae.get_latent_representation()
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_scVI")
sc.tl.umap(adata)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()
print("scVI done")

ModuleNotFoundError: No module named 'scvi'

In [8]:
# 1.4 scVI UMAP: did the donors mix?
sc.pl.embedding(adata, basis="umap_scvi", color=["donor_id", "cell_type_broad"], wspace=0.5)

NameError: name 'sc' is not defined

## 1.5 · Harmony, a fair baseline  *(provided)*

In [9]:
import harmonypy
ho = harmonypy.run_harmony(adata.obsm["X_pca"].astype(np.float64), adata.obs, "donor_id")
Z = np.asarray(ho.Z_corr)
adata.obsm["X_pca_harmony"] = Z if Z.shape[0] == adata.n_obs else Z.T
print("Harmony done")

ModuleNotFoundError: No module named 'harmonypy'

## 1.6 · Judge the fix
**TODO:** compute how well each method (no integration, Harmony, scVI) mixes donors and preserves cell types, then decide which you would use.

In [11]:
from sklearn.metrics import silhouette_score
# 1.6 Score each embedding.
# Batch silhouette (by donor): closer to 0 or below = donors well mixed (lower is better)
# Bio silhouette (by cell type): higher = cell types stay separated (higher is better)
idx = rng.choice(adata.n_obs, size=min(5000, adata.n_obs), replace=False)
reps = {"No integration": "X_pca", "Harmony": "X_pca_harmony", "scVI": "X_scVI"}
rows = []
for name, key in reps.items():
    emb = adata.obsm[key][idx]
    batch = silhouette_score(emb, adata.obs["donor_id"].values[idx])
    bio   = silhouette_score(emb, adata.obs["cell_type_broad"].values[idx])
    rows.append({"method": name, "batch_silhouette (lower=better)": round(batch, 3),
                 "bio_silhouette (higher=better)": round(bio, 3)})
scores = pd.DataFrame(rows).set_index("method")
scores

NameError: name 'rng' is not defined

**TODO (interpretation):** Which method would you use, and why? Does the expensive deep-learning method clearly beat the simple one?

*Your answer here.*

## 1.7 · Annotate and confirm with markers
The broad cell types are already assigned. **TODO:** plot the labeled map and a few canonical markers to confirm it.

In [12]:
# 1.7 Labeled map + canonical markers
sc.pl.embedding(adata, basis="umap_scvi", color="cell_type_broad")
# CD3D = T cells, MS4A1 = B cells, CD14 = monocytes, EPCAM = epithelial/tumor,
# PECAM1 = endothelial, COL1A1 = fibroblasts
markers = [g for g in ["CD3D", "MS4A1", "CD14", "EPCAM", "PECAM1", "COL1A1"] if g in adata.raw.var_names]
sc.pl.embedding(adata, basis="umap_scvi", color=markers, use_raw=True, ncols=3)

NameError: name 'sc' is not defined

## 1.8 · Locate your target in the cell types
**TODO:** choose your target gene, plot it, and identify which cell type(s) express it.

In [13]:
TARGET = "ERBB2"   # choose: ERBB2, ESR1, EPCAM, MKI67, PTPRC, CD68, CD3D
assert TARGET in adata.raw.var_names, f"{TARGET} not in the single-cell data; pick another"

# 1.8 Target on the scVI UMAP
sc.pl.embedding(adata, basis="umap_scvi", color=[TARGET, "cell_type_broad"], use_raw=True, wspace=0.5)

# Which cell types express it? Dot plot + a table (mean expression, % of cells expressing)
sc.pl.dotplot(adata, [TARGET], groupby="cell_type_broad", use_raw=True)
expr = adata.raw[:, TARGET].X
expr = expr.toarray().ravel() if hasattr(expr, "toarray") else np.asarray(expr).ravel()
target_by_type = (pd.DataFrame({"cell_type": adata.obs["cell_type_broad"].values, "expr": expr})
                  .groupby("cell_type", observed=True)["expr"]
                  .agg(mean_expr="mean", pct_expressing=lambda x: 100 * (x > 0).mean(), n_cells="size")
                  .sort_values("mean_expr", ascending=False).round(3))
target_by_type

NameError: name 'adata' is not defined

**TODO (interpretation):** Which cell type(s) express your target? 

*Your answer here.*

---
# PART 2 · Spatial: where those cells sit in the tumor

## 2.1 · Load the provided Visium section  *(provided)*
A real 10x human breast-cancer Visium section. Downloads once on the first run.

In [14]:
spatial = sc.datasets.visium_sge(sample_id="V1_Breast_Cancer_Block_A_Section_1")
spatial.var_names_make_unique()
print(spatial)
print("spots x genes:", spatial.shape)

NameError: name 'sc' is not defined

## 2.2 · Pipeline and clustering  *(provided)*

In [15]:
spatial.var["mt"] = spatial.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(spatial, qc_vars=["mt"], inplace=True)
sc.pp.filter_cells(spatial, min_counts=500)
spatial.layers["counts"] = spatial.X.copy()
sc.pp.normalize_total(spatial, target_sum=1e4)
sc.pp.log1p(spatial)
spatial.raw = spatial
sc.pp.highly_variable_genes(spatial, n_top_genes=2000)
sc.pp.pca(spatial)
sc.pp.neighbors(spatial)
sc.tl.leiden(spatial, resolution=0.6, key_added="cluster")
print("spatial domains found:", spatial.obs["cluster"].nunique())

NameError: name 'spatial' is not defined

## 2.3 · Spatial domains on the tissue
**TODO:** draw the clusters back onto the tissue and relate them to tumor, stroma, and immune regions.

In [16]:
# 2.3 Spatial domains drawn back on the H&E image
sq.pl.spatial_scatter(spatial, color="cluster")

NameError: name 'sq' is not defined

In [ ]:
# 2.3b Label each spatial domain with marker signatures (tumor / stroma / immune)
signatures = {
    "tumor_epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "CDH1", "ERBB2"],
    "stroma":           ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "FN1"],
    "immune":           ["PTPRC", "CD74", "HLA-DRA", "CD3D", "IGKC", "LYZ"],
}
for name, genes in signatures.items():
    genes = [g for g in genes if g in spatial.raw.var_names]
    sc.tl.score_genes(spatial, genes, score_name=name, use_raw=True)
domain_sig = spatial.obs.groupby("cluster", observed=True)[list(signatures)].mean().round(3)
domain_sig["n_spots"] = spatial.obs["cluster"].value_counts().sort_index()
domain_sig["likely_region"] = domain_sig[list(signatures)].idxmax(axis=1)
print(domain_sig)
spatial.obs["region"] = spatial.obs["cluster"].map(domain_sig["likely_region"]).astype("category")
sq.pl.spatial_scatter(spatial, color=list(signatures) + ["region"], ncols=2)

**TODO (interpretation):** Do the domains line up with recognizable tissue architecture (tumor nests, stroma, immune areas)?

*Your answer here.*

## 2.4 · Spatially variable genes (Moran's I)
The spatial neighbor graph is built for you. **TODO:** compute Moran's I, list the top genes, and plot a few.

In [17]:
sq.gr.spatial_neighbors(spatial)

NameError: name 'sq' is not defined

In [18]:
# 2.4 Moran's I (uses the highly variable genes by default)
sq.gr.spatial_autocorr(spatial, mode="moran")
top = spatial.uns["moranI"].head(8)
print(top[["I", "pval_norm"]])
sq.pl.spatial_scatter(spatial, color=spatial.uns["moranI"].head(3).index.tolist())

NameError: name 'sq' is not defined

## 2.5 · Neighborhood enrichment
**TODO:** compute which domains sit next to each other more than chance, and read off any tumor-immune interface.

In [19]:
# 2.5 Neighborhood enrichment: which domains touch more than chance (high z-score)?
sq.gr.nhood_enrichment(spatial, cluster_key="cluster")
sq.pl.nhood_enrichment(spatial, cluster_key="cluster")

NameError: name 'sq' is not defined

**TODO (interpretation):** Which domains interface more than chance? Is there a tumor-immune boundary?

*Your answer here.*

## 2.6 · Locate your target in the tissue
**TODO:** color the tissue by the same target gene and describe where it sits.

In [20]:
# 2.6 Same target on the tissue, next to the domains
if TARGET in spatial.raw.var_names:
    sq.pl.spatial_scatter(spatial, color=[TARGET, "cluster"])
    # Which domains carry the target? (mean log-expression per domain)
    t = spatial.raw[:, TARGET].X
    t = t.toarray().ravel() if hasattr(t, "toarray") else np.asarray(t).ravel()
    print(pd.Series(t, index=spatial.obs_names).groupby(spatial.obs["cluster"], observed=True)
          .mean().sort_values(ascending=False).round(3))
else:
    print(TARGET, "not in this panel; pick a gene present in spatial.raw.var_names")

NameError: name 'spatial' is not defined

**TODO (interpretation):** Where does your target sit: tumor core, invasive margin, or an immune niche?

*Your answer here.*

---
# PART 3 · Connect it (your report)

Write a 4–5 page report (separate PDF) that ties the two views together:

- Which cell types express your target (Part 1), and where those cells sit in the tumor (Part 2).
- Your integration choice, justified with the scores.
- The spatial context and why it matters clinically (a target at the invasive margin versus shut inside the core).
- The limitations of each analysis: donor count and label resolution (single-cell); spot resolution, panel, and segmentation (spatial).

Then **Kernel → Restart & Run All**, commit this notebook and your report to your biot6900 repo, and post the repo link on Canvas.

---
## Results summary (for the report)
Writes every key number to `module8_results_summary.txt`.

In [ ]:
# Summary dump: all key numbers in one text file (for the report)
import io
buf = io.StringIO()
def w(*a): print(*a, file=buf)
w("TARGET:", TARGET)
w("\n== PART 1 ==")
w(f"cells after QC: {adata.n_obs} | donors: {adata.obs['donor_id'].nunique()}")
w("assays:\n", adata.obs['assay'].value_counts().to_string())
w("broad cell types:\n", adata.obs['cell_type_broad'].value_counts().to_string())
w("integration scores:\n", scores.to_string())
w("target by cell type:\n", target_by_type.to_string())
w("\n== PART 2 ==")
w("spots:", spatial.n_obs)
w("domain signatures:\n", domain_sig.to_string())
w("Moran's I top 15:\n", spatial.uns['moranI'].head(15)[['I','pval_norm']].to_string())
z = spatial.uns['cluster_nhood_enrichment']['zscore']
cats = spatial.obs['cluster'].cat.categories
w("nhood enrichment z-scores:\n", pd.DataFrame(z, index=cats, columns=cats).round(1).to_string())
if TARGET in spatial.raw.var_names:
    t = spatial.raw[:, TARGET].X
    t = t.toarray().ravel() if hasattr(t, 'toarray') else np.asarray(t).ravel()
    w("target by domain:\n", pd.Series(t, index=spatial.obs_names).groupby(spatial.obs['cluster'], observed=True)
      .agg(['mean', lambda x: 100*(x>0).mean()]).round(3).to_string())
open("module8_results_summary.txt", "w").write(buf.getvalue())
print(buf.getvalue())